# M17 — Context-Aware Athletic Signatures (Colab)

**Status: tasks T05, T12–T14 authored — never executed by the author.**

## A. Introduction

- **Question.** Do athletes have repeatable movement characteristics that can be separated from movement context and that transfer from Combine to NFL observations? Latent factors are never called biological abilities without independent validation.
- **Why.** M14 [EXP]: 1 of 30 metric instances passed its repeatability screen (e.g. braking efficiency ICC ≈ 0.22, turn-speed retention ≈ −0.11). Low ICC can be real variability, poor event definitions, or mixed contexts — this milestone separates the three before learning any trait.
- **Tracks.** A event diagnosis (failure modes, per-drill vs pooled ICC, threshold grid), a-priori context strata, censoring-aware recovery · B context-adjusted athlete effects (MixedLM or method of moments, fold-safe BLUPs, Combine ↔ NFL correlation) · C fusion of G15-eligible representations · D alignment (Procrustes → CORAL → OT → context-matched InfoNCE) with damage checks · E signatures and **G17**.
- **Inputs (read only).** Bundle; M14 detector (code); M15 run (sequence features, embeddings, G15 gate); M16 run (NFL events, targets, freeze, analysis frame, leakage verdict).
- **Simple first.** Every advanced arm is compared with its simple benchmark (concat for fusion, Procrustes for alignment); neural InfoNCE needs both training opt-ins.
- **Statuses.** COMPLETED · CACHED · BLOCKED_* · FAILED_VALIDATION · FAILED · NOT_RUN; G17 defaults to NOT_EVALUATED.

## B. Setup

In [ ]:
# Setup 1 — mount Drive, install nflbowl (Colab PyTorch / CUDA kept); statsmodels / POT optional
from google.colab import drive
drive.mount('/content/drive')
REPO = 'git+https://github.com/mybenkhadda/NFL_bowl.git@main'
%pip install -q {REPO}
%pip install -q --upgrade --force-reinstall --no-deps {REPO}
%pip install -q statsmodels POT markdown

In [ ]:
# Setup 2 — imports (after a reinstall: Runtime → Restart session, then continue from here)
import nflbowl
print('nflbowl', nflbowl.__version__)
assert tuple(int(x) for x in nflbowl.__version__.split('.')[:2]) >= (0, 7)
import numpy as np
import polars as pl
from IPython.display import Markdown, display
from nflbowl.lab import handoff
from nflbowl.lab import status as S
from nflbowl.lab.m15 import optional as m15opt
from nflbowl.lab.m16 import pipeline as m16pipe
from nflbowl.lab.m17 import config as m17cfg, pipeline as m17pipe, plots as m17plots
display(m15opt.report())

In [ ]:
# Setup 3 — configuration (the ONLY cell to edit). Strata boundaries are fixed here, before any analysis.
CFG = {
    'experiment_name': 'm17_context_athletic_signatures_v1',
    'bundle_dir': '/content/drive/MyDrive/nfl-big-data-bowl-2027/bundles/m12f_v1_clean',
    'drive_root': '/content/drive/MyDrive/NFLBowl',
    'entry_speed_bins': [2.0, 4.0, 6.0, 8.0], 'turn_bins_deg': [20.0, 90.0],
    'run_training': False, 'allow_new_training': False,      # context-matched InfoNCE only
    'allow_descriptive': False,
    'seed': 42,
}
cfg = m17cfg.resolve(CFG)
run = m17cfg.M17Run(cfg.run_dir, cfg)
run.snapshot_config(cfg, handoff.environment(cfg.m13()))
print('run dir', run.root, '· fingerprint', cfg.fingerprint())
display(run.ledger.table())

In [ ]:
# Setup 4 — experiment (bundle + folds), M15 data view and E15.A1 features (read from the M15 run when present)
from pathlib import Path

def step_open(ctx):
    return handoff.open_experiment(cfg.m13(), run)

EXP = S.run_step(run, run.ledger, 'E17.open', step_open, note='open the M17 experiment on the official bundle / science')
if EXP is None:
    raise SystemExit('the experiment could not be opened — see the status ledger and logs/')
CFG15 = cfg.m15()
DATA = m16pipe.data_from_experiment(EXP, CFG15)
p = Path(cfg.m15_run_dir) / 'features' / 'signal' / 'sequence_time_features.parquet'
if p.is_file():
    FEAT = pl.read_parquet(p)                                   # M15 E15.A1 output (read only)
else:
    from nflbowl.lab.m15.signals import feature_table
    FEAT = feature_table(DATA, CFG15)                           # recomputed (same code) when M15 has not run
FOLDS = sorted(set(EXP.folds_by_athlete.values()))
print('sequences', DATA.index.height, '· folds', FOLDS, '· features from', 'M15 run' if p.is_file() else 'recomputation')

## C. Track A — E17.A1 why did M14 event metrics fail? (T05)

M14 detector reused unchanged; one-factor-at-a-time variation of `brake_thresh`, `sustain_frames`, `drop_frac` (Combine rows all, NFL rows on a seeded sample; each variant cached). Read-outs: flag shares per domain × kind, ICC per drill vs pooled agility (kind-centred), best variant, and a per-metric diagnosis (drill heterogeneity · definition-sensitive · low under every definition).

In [ ]:
def step_a1(ctx):
    return m17pipe.run_a1(cfg, DATA, run, ctx)

A1 = S.run_step(run, run.ledger, 'E17.A1', step_a1, note='failure modes + per-drill ICC + threshold grid', requires=['E17.open'])
if A1 is not None:
    display(A1['failure_modes'])
    display(Markdown('**Diagnosis per metric**')); display(A1['diagnosis'])

In [ ]:
if A1 is not None:
    fig, dat = m17plots.failure_bars(A1['failure_modes'])
    run.figure(fig, 'M17_A1_01_failure_modes', data=dat, question='Why do detected events fail completeness?', caption='Detector flag shares by domain and kind.')
    fig, dat = m17plots.drill_vs_pooled_icc(A1['icc'])
    run.figure(fig, 'M17_A1_02_drill_vs_pooled_icc', data=dat, question='Is low repeatability caused by pooling drills?', caption='ICC per drill (blue) vs pooled agility (orange).')

## E17.A2 / A3 — context strata, coverage and censoring-aware recovery

Strata fixed a priori (entry speed × turn class; deceleration demand recorded). A stratum is *supported* only with ≥ `min_stratum_athletes` athletes having ≥ 2 events in **both** domains. Recovery time is right-censored when the clip ends first: Kaplan–Meier athlete medians vs the naive completed-only median, compared on split-half reliability.

In [ ]:
def step_a2(ctx):
    if A1 is None:
        raise S.ResearchBlocked(S.BLOCKED_MISSING_INPUT, 'E17.A1 events not available this session', missing=['E17.A1'])
    ev, note = m17pipe.all_events(cfg, A1['events'])
    print(note)
    return m17pipe.run_a2(cfg, ev, run, ctx)

A2 = S.run_step(run, run.ledger, 'E17.A2', step_a2, note='context strata + coverage', requires=['E17.A1'])
if A2 is not None:
    display(A2['coverage'])

def step_a3(ctx):
    return m17pipe.run_a3(cfg, A2['strata'], run, ctx)

A3 = S.run_step(run, run.ledger, 'E17.A3', step_a3, note='censoring-aware recovery', requires=['E17.A2']) if A2 is not None else None
if A3 is not None:
    display(A3['reliability'])

In [ ]:
if A2 is not None:
    fig, dat = m17plots.stratum_coverage(A2['coverage'])
    run.figure(fig, 'M17_A2_01_stratum_coverage', data=dat, question='Where are Combine and NFL events comparable?', caption='Athletes per stratum and domain; supported strata labelled.')

## E. Track B — E17.B1 / B2 context-adjusted athlete effects (T12)

`y = μ + context (kind × stratum) + athlete + noise` per metric and domain; parameters for the athletes of fold *f* come from the other folds only; MixedLM (REML) when available, else method of moments (recorded per row). Variance shares, k-event reliability, BLUP shrinkage and the disattenuated Combine ↔ NFL correlation within position.

In [ ]:
def step_b(ctx):
    if A2 is None:
        raise S.ResearchBlocked(S.BLOCKED_MISSING_INPUT, 'strata not available (run E17.A2)', missing=['E17.A2'])
    return m17pipe.run_b(cfg, A2['strata'], EXP.folds_by_athlete, run, ctx)

B = S.run_step(run, run.ledger, 'E17.B', step_b, note='variance components + BLUPs + cross-domain r', requires=['E17.A2'])
if B is not None:
    display(B['summary'])
    display(Markdown('**Combine ↔ NFL athlete effects**')); display(B['cross_domain'])

In [ ]:
if B is not None and B['summary'].height:
    fig, dat = m17plots.variance_stacked(B['summary'])
    run.figure(fig, 'M17_B_01_variance_components', data=dat, question='How much variation is athlete, context, noise?', caption='Shares of total variance per metric and domain.')
    fig, dat = m17plots.reliability_vs_k(B['summary'])
    run.figure(fig, 'M17_B_02_reliability_vs_k', data=dat, question='How many events give a reliable athlete effect?', caption='σ²u / (σ²u + σ²e / k) for Combine events.')
    for m in B['cross_domain']['metric'].to_list()[:4] if B['cross_domain'].height else []:
        fig, dat = m17plots.combine_vs_nfl(B['effects'], B['cross_domain'], m)
        run.figure(fig, f'M17_B_03_combine_vs_nfl_{m}', data=dat, question='Do Combine and NFL athlete effects agree?', caption=f'{m}: shrunk athlete effects.')
        fig, dat = m17plots.caterpillar(B['effects'], m)
        run.figure(fig, f'M17_B_04_caterpillar_{m}', data=dat, question='How spread are athlete effects within position?', caption=f'{m}: BLUPs by position.')

## E17.C1 / D1 — fusion and context-conditioned alignment (T13; conditional on G15 and the M16 leakage audit)

Run only for representations with G15 = GO that passed the M16 leakage audit (otherwise BLOCKED_MISSING_INPUT — the expected outcome when no representation advanced). Fusion arms: concat, reliability-weighted (train-fold ICC); adoption needs ≥ 4/5 folds better than the best single representation. Alignment: Procrustes → CORAL → entropic OT → context-matched InfoNCE (opt-in); adoption over Procrustes needs ≥ 4/5 folds and no damage.

In [ ]:
def step_c(ctx):
    return m17pipe.run_c(cfg, EXP.bundle.athletes, FOLDS, run, ctx)

C = S.run_step(run, run.ledger, 'E17.C', step_c, note='representation fusion', requires=['E17.open'], enabled=cfg.run_fusion)
if C is not None:
    print('pair:', C['pair']); display(C['adoption'])

def step_d(ctx):
    return m17pipe.run_d(cfg, EXP.bundle.athletes, FEAT, FOLDS, run, ctx)

D = S.run_step(run, run.ledger, 'E17.D', step_d, note='alignment arms + damage checks', requires=['E17.open'], enabled=cfg.run_alignment)
if D is not None:
    display(D['adoption'])

In [ ]:
if D is not None and D['summary'].height:
    fig, dat = m17plots.alignment_tradeoff(D['summary'], D['damage'])
    run.figure(fig, 'M17_D_01_alignment_tradeoff', data=dat, question='Does aligning domains help athlete identification?',
               caption='Same-position MRR vs MMD² per arm (fold means); outlined = destructive.')

## E17.E1 — signatures and G17 (T14)

Candidate = Combine context-adjusted athlete effect per metric. Criteria a–f (repeatability after context, athlete share of total, novelty vs M1 + position, held-out split-half, within-position association with an eligible M16 target after BH, a-priori physical anchor for the label). Only `validated` signatures enter M18 rung B7.

In [ ]:
def step_e(ctx):
    if B is None or A2 is None:
        raise S.ResearchBlocked(S.BLOCKED_MISSING_INPUT, 'athlete effects not available (run E17.B)', missing=['E17.B'])
    return m17pipe.run_e(cfg, B, A2['strata'], EXP.folds_by_athlete, run, ctx)

E = S.run_step(run, run.ledger, 'G17', step_e, note='signature criteria + G17', requires=['E17.B'])
if E is not None:
    display(E['gate'].filter(pl.col('criterion') == 'G17 decision'))
    print('validated signature columns for M18 B7:', E['validated'])
    fig, dat = m17plots.gate_matrix(E['gate'], 'G17')
    run.figure(fig, 'M17_E_01_gate_matrix', data=dat, question='Which signatures are valid athletic measurements?', caption='G17 criteria per signature.')

## F / G. Results, persistence and hand-off

Pivot rules: repaired events still unreliable → the braking traits are not resolvable at 10 Hz with ≈ 2 trials (a reportable negative); athlete share < noise → no trait claim, B7 blocked; Combine ↔ NFL r ≈ 0 → no transfer for that family; destructive alignment → keep unaligned representations.

In [ ]:
run.write_status()
run.save_markdown(run.handoff_markdown(), 'm17_handoff')
display(run.ledger.table())
print('SEND BACK: manifests/stages.json, manifests/m17_status.json, logs/*_traceback.txt, tables/events_v2/diagnosis.csv, tables/events_v2/per_drill_icc.csv,')
print('           tables/context/stratum_coverage.csv, tables/variance/summary.csv, tables/variance/cross_domain.csv, tables/synthesis/m17_gate.csv')